## Cell 0: setup and miRNA seed families

In [1]:
# 07_graph - Cell 0: setup and miRNA seed families from the raw ENCORI file
from pathlib import Path
import json, yaml
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
with open(PROJECT_ROOT / "configs" / "paths.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
D = PROJECT_ROOT / cfg["outputs"]["processed_dir"] / "dataset_v1"
G = D / "graph"; G.mkdir(exist_ok=True)
ENC_MIRNA = next((PROJECT_ROOT / cfg["datasets"]["encori_mirna"]["dir"]).glob("*.txt"))

# 1. Read miRNA id, name and aligned sequence (chunks; three columns only)
cols = ["miRNAid", "miRNAname", "miRseq"]
found = set()
for ch in pd.read_csv(ENC_MIRNA, sep="\t", comment="#", usecols=cols, dtype=str, chunksize=500_000):
    found.update(map(tuple, ch.drop_duplicates().to_numpy()))
mirs = pd.DataFrame(sorted(found), columns=["mirna_id", "mirna_name", "mirseq"])

# 2. ENCORI writes miRseq as aligned, 3'->5', with gaps and case: clean, then reverse to 5'->3'
REVERSED = True                                                            # confirmed with let-7a-5p and miR-21-5p
mirs["mirseq"] = (mirs["mirseq"].str.upper().str.replace("T", "U")
                  .str.replace(r"[^ACGU]", "", regex=True))               # drop gaps; case marks pairing only
if REVERSED:
    mirs["mirseq"] = mirs["mirseq"].str[::-1]
n_forms = mirs.groupby("mirna_id")["mirseq"].nunique()
print(f"miRNAs still with >1 sequence after cleaning: {int((n_forms > 1).sum())} of {len(n_forms)}")
mirs = (mirs.groupby(["mirna_id", "mirna_name"])["mirseq"]
        .agg(lambda s: s.value_counts().index[0]).reset_index())          # one sequence per miRNA
assert mirs["mirna_id"].is_unique, "a miRNA still has more than one sequence"

# Check against miRBase mature sequences (5'->3')
for name, ref in {"hsa-let-7a-5p": "UGAGGUAGUAGGUUGUAUAGUU", "hsa-miR-21-5p": "UAGCUUAUCAGACUGAUGUUGA"}.items():
    got = mirs.loc[mirs["mirna_name"] == name, "mirseq"]
    assert got.empty or got.iat[0] == ref, f"{name}: {got.iat[0]} != {ref}"

# 3. Seed = mature nucleotides 2-8; miRNAs with the same seed form one family
mirs["seed"] = mirs["mirseq"].str[1:8]
fam = mirs.groupby("seed")["mirna_name"].apply(lambda s: "|".join(sorted(s)))
mirs["family"] = mirs["seed"].map(fam)
print(f"miRNAs {len(mirs)} | seed families {mirs['seed'].nunique()} | "
      f"families with >1 member {int((mirs.groupby('seed').size() > 1).sum())}")

# 4. Every miRNA in our edge file must have a family
edges_m = pd.read_csv(D / "mirna_edges.csv", usecols=["mirna_id"])
missing = set(edges_m["mirna_id"]) - set(mirs["mirna_id"])
assert not missing, f"{len(missing)} miRNAs in mirna_edges.csv have no sequence"
let7 = mirs.loc[mirs["mirna_name"] == "hsa-let-7a-5p", "family"]
print("let-7a family:", let7.iat[0] if len(let7) else "not present")
print("examples of merged families:", mirs.loc[mirs["family"].str.contains(r"\|"), "family"].drop_duplicates().head(5).tolist())
mirs.to_csv(G / "mirna_families.csv", index=False)

miRNAs still with >1 sequence after cleaning: 0 of 642
miRNAs 642 | seed families 421 | families with >1 member 125
let-7a family: hsa-let-7a-5p|hsa-let-7b-5p|hsa-let-7c-5p|hsa-let-7d-5p|hsa-let-7e-5p|hsa-let-7f-5p|hsa-let-7g-5p|hsa-let-7i-5p|hsa-miR-4458|hsa-miR-4500|hsa-miR-98-5p
examples of merged families: ['hsa-let-7a-5p|hsa-let-7b-5p|hsa-let-7c-5p|hsa-let-7d-5p|hsa-let-7e-5p|hsa-let-7f-5p|hsa-let-7g-5p|hsa-let-7i-5p|hsa-miR-4458|hsa-miR-4500|hsa-miR-98-5p', 'hsa-miR-15a-5p|hsa-miR-15b-5p|hsa-miR-16-5p|hsa-miR-195-5p|hsa-miR-424-5p|hsa-miR-497-5p|hsa-miR-6838-5p', 'hsa-miR-106a-5p|hsa-miR-106b-5p|hsa-miR-17-5p|hsa-miR-20a-5p|hsa-miR-20b-5p|hsa-miR-519d-3p|hsa-miR-526b-3p|hsa-miR-93-5p', 'hsa-miR-18a-5p|hsa-miR-18b-5p|hsa-miR-4735-3p', 'hsa-miR-19a-3p|hsa-miR-19b-3p']


### check for diagnisis 

In [2]:
# Diagnose: why do some miRNAs have more than one sequence?
multi = mirs.groupby("mirna_id")["mirseq"].nunique()
print(f"miRNAs with >1 sequence: {int((multi > 1).sum())} of {mirs['mirna_id'].nunique()}")
ex = mirs[mirs["mirna_id"].isin(multi[multi > 1].index[:4])].sort_values("mirna_id")
print(ex.to_string(index=False))
print("characters used:", sorted(set("".join(mirs["mirseq"]))))
print("length min/median/max:", mirs["mirseq"].str.len().quantile([0, .5, 1]).tolist())

# Orientation check with well-known mature sequences (miRBase, written 5'->3')
known = {"hsa-let-7a-5p": "UGAGGUAGUAGGUUGUAUAGUU", "hsa-miR-21-5p": "UAGCUUAUCAGACUGAUGUUGA"}
for name, ref in known.items():
    got = mirs.loc[mirs["mirna_name"] == name, "mirseq"].unique()[:3]
    print(f"{name}: ENCORI {list(got)} | reference 5'->3' {ref}")

miRNAs with >1 sequence: 642 of 642
    mirna_id    mirna_name                        mirseq
MIMAT0000062 hsa-let-7a-5p     uuGA-UA-UGUU-GGAUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p      uugaUAUGUUGG-A-UGAUGGAGu
MIMAT0000062 hsa-let-7a-5p    uugaUAUGUUGG-A---UGAUGGAGu
MIMAT0000062 hsa-let-7a-5p      uugaUAUGUUGG--AUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p     uugaUAUGUUGG--AU-GAUGGAGu
MIMAT0000062 hsa-let-7a-5p    uugaUAUGUUGG--AU--GAUGGAGu
MIMAT0000062 hsa-let-7a-5p     uugaUAUGUUGG---AUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p    uugaUAUGUUGG----AUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p       uugaUAUGUUG-GAUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p      uugaUAUGUUG-GAU-GAUGGAGu
MIMAT0000062 hsa-let-7a-5p      uugaUAUGUUG--GAUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p   uugaUAUGUUGG-AU----GAUGGAGu
MIMAT0000062 hsa-let-7a-5p     uugaUAUGUUG---GAUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p      uugaUAUGUU-GGAU-GAUGGAGu
MIMAT0000062 hsa-let-7a-5p      uugaUAUGUU--GGAUGAUGGAGu
MIMAT0000062 hsa-let-7a-5p    uugaUAUGUU--GGAU--GAUG

In [4]:
# after the REVERSED step, before taking the most common sequence:
n_forms = mirs.groupby("mirna_id")["mirseq"].nunique()
print(f"miRNAs still with >1 sequence after normalising: {int((n_forms > 1).sum())} of {len(n_forms)}")

miRNAs still with >1 sequence after normalising: 0 of 642


## Cell 1: node tables, edge tables and circRNA features

In [2]:
# 07_graph - Cell 1: node tables, edge tables and circRNA features (all indices 0-based)
EMB_DIR = D / "RNA-FM_embedding_output"                          # same folder as probe.yaml embeddings.dir
circ = pd.read_csv(D / "circrnas.csv", keep_default_na=False)
emb_idx = pd.read_csv(EMB_DIR / "rnafm_embeddings_index.csv")
assert emb_idx["circbase_id"].tolist() == circ["circbase_id"].tolist(), "embedding rows out of order"
C_ID = {c: i for i, c in enumerate(circ["circbase_id"])}

# circRNA features: RNA-FM layer 12 (overlapping windows) + log10 length; raw (standardised per fold later)
emb = np.load(EMB_DIR / "rnafm_embeddings_overlap.npy")
X = np.hstack([emb, np.log10(circ["seq_len"].to_numpy())[:, None]]).astype(np.float32)
assert X.shape == (len(circ), 641) and np.isfinite(X).all()

# disease nodes and circRNA-disease edges
dis = pd.read_csv(D / "diseases.csv")
assert dis["disease_id"].tolist() == list(range(len(dis)))
D_ID = dict(zip(dis["disease"], dis["disease_id"]))
cda = pd.read_csv(D / "cda_pairs.csv")
e_cd = pd.DataFrame({"src": cda["circbase_id"].map(C_ID), "dst": cda["disease"].map(D_ID)})
assert e_cd.notna().all().all() and not e_cd.duplicated().any() and len(e_cd) == 2_137

# miRNA family nodes and circRNA-family edges (working rule, collapsed to families)
fams = pd.read_csv(G / "mirna_families.csv")
mir = pd.read_csv(D / "mirna_edges.csv", dtype={"pos_working": bool})
mir = mir[mir["pos_working"]]
assert len(mir) == 13_077, f"working miRNA edges: {len(mir)}"
mir = mir.merge(fams[["mirna_id", "seed", "family"]], on="mirna_id", how="left")
assert mir["seed"].notna().all(), "a miRNA without a family"
fam_nodes = (mir.groupby("seed").agg(family=("family", "first"), n_circ=("circbase_id", "nunique"))
             .reset_index().sort_values("seed").reset_index(drop=True))
fam_nodes.insert(0, "fam_idx", range(len(fam_nodes)))
F_ID = dict(zip(fam_nodes["seed"], fam_nodes["fam_idx"]))
e_cm = pd.DataFrame({"src": mir["circbase_id"].map(C_ID), "dst": mir["seed"].map(F_ID)}).drop_duplicates()
assert e_cm.notna().all().all()

# RBP nodes and circRNA-RBP edges (working rule)
rbp = pd.read_csv(D / "rbp_edges.csv", keep_default_na=False, dtype={"pos_working": bool})
rbp = rbp[rbp["pos_working"]]
rbp_nodes = (rbp.groupby("rbp")["circbase_id"].nunique().rename("n_circ")
             .reset_index().sort_values("rbp").reset_index(drop=True))
rbp_nodes.insert(0, "rbp_idx", range(len(rbp_nodes)))
R_ID = dict(zip(rbp_nodes["rbp"], rbp_nodes["rbp_idx"]))
e_cr = pd.DataFrame({"src": rbp["circbase_id"].map(C_ID), "dst": rbp["rbp"].map(R_ID)}).drop_duplicates()
assert e_cr.notna().all().all() and len(e_cr) == 22_578, f"working RBP edges: {len(e_cr)}"

for e in (e_cd, e_cm, e_cr):
    e[["src", "dst"]] = e[["src", "dst"]].astype(np.int64)

# Summary
deg = lambda e: np.bincount(e["src"], minlength=len(circ))
d_m, d_r = deg(e_cm), deg(e_cr)
no_aux = (d_m == 0) & (d_r == 0)
print(f"nodes   circRNA {len(circ)} | disease {len(dis)} | miRNA family {len(fam_nodes)} | RBP {len(rbp_nodes)}")
print(f"edges   circRNA-disease {len(e_cd)} | circRNA-miRNA family {len(e_cm)} (from {len(mir)} miRNA edges) | circRNA-RBP {len(e_cr)}")
print(f"circRNAs with a miRNA edge {int((d_m > 0).sum())} | with an RBP edge {int((d_r > 0).sum())} | "
      f"with NO auxiliary edge {int(no_aux.sum())} ({100 * no_aux.mean():.1f}%)")
print(f"median degree  miRNA family {int(np.median(d_m[d_m > 0]))} | RBP {int(np.median(d_r[d_r > 0]))}")

# Save: circRNA table with edge counts, status, and detailed status
circ_out = circ.assign(circ_idx=range(len(circ)), n_mirna_edges=d_m, n_rbp_edges=d_r)
circ_out["aux_status"] = np.select([(d_m > 0) & (d_r > 0), d_m > 0, d_r > 0],
                                   ["both", "miRNA_only", "RBP_only"], default="neither")

c_lab = pd.read_csv(D / "circrnas.csv", keep_default_na=False, dtype={"mirna_labelled": bool, "rbp_labelled": bool})
assert c_lab["circbase_id"].tolist() == circ_out["circbase_id"].tolist()
in_any = (c_lab["mirna_labelled"] | c_lab["rbp_labelled"]).to_numpy()
circ_out["aux_status_detail"] = np.where(circ_out["aux_status"] != "neither", circ_out["aux_status"],
                                         np.where(in_any, "neither_measured", "neither_not_in_encori"))

circ_out[["circ_idx", "circbase_id", "host_group", "seq_len", "n_diseases", "is_hub",
          "n_mirna_edges", "n_rbp_edges", "aux_status", "aux_status_detail"]].to_csv(G / "nodes_circrna.csv", index=False)
print(circ_out["aux_status_detail"].value_counts().to_string())

dis.to_csv(G / "nodes_disease.csv", index=False)
fam_nodes.to_csv(G / "nodes_mirna_family.csv", index=False)
rbp_nodes.to_csv(G / "nodes_rbp.csv", index=False)
e_cd.to_csv(G / "edges_circrna_disease.csv", index=False)
e_cm.to_csv(G / "edges_circrna_mirna.csv", index=False)
e_cr.to_csv(G / "edges_circrna_rbp.csv", index=False)
np.save(G / "x_circrna.npy", X)
(G / "graph_tables.json").write_text(json.dumps({
    "nodes": {"circRNA": len(circ), "disease": len(dis), "miRNA_family": len(fam_nodes), "RBP": len(rbp_nodes)},
    "edges": {"circRNA-disease": len(e_cd), "circRNA-miRNA_family": len(e_cm), "circRNA-RBP": len(e_cr)},
    "circRNA_features": "RNA-FM L12 overlap (640) + log10 length (1), raw",
    "circRNAs_without_auxiliary_edges": int(no_aux.sum()),
    "circRNA_aux_status": {k: int(v) for k, v in circ_out["aux_status"].value_counts().items()},
    "circRNA_aux_status_detail": {k: int(v) for k, v in circ_out["aux_status_detail"].value_counts().items()},
}, indent=2))

nodes   circRNA 1582 | disease 207 | miRNA family 416 | RBP 225
edges   circRNA-disease 2137 | circRNA-miRNA family 7540 (from 13077 miRNA edges) | circRNA-RBP 22578
circRNAs with a miRNA edge 835 | with an RBP edge 1314 | with NO auxiliary edge 256 (16.2%)
median degree  miRNA family 5 | RBP 12
aux_status_detail
both                     823
RBP_only                 491
neither_not_in_encori    148
neither_measured         108
miRNA_only                12


594

In [8]:
print(circ_out.groupby("aux_status")["seq_len"].median())

aux_status
RBP_only      411.0
both          643.0
miRNA_only    373.5
neither       461.5
Name: seq_len, dtype: float64


In [10]:
c = pd.read_csv(D / "circrnas.csv", keep_default_na=False, dtype={"mirna_labelled": bool, "rbp_labelled": bool})
nb = circ_out["aux_status"].eq("neither").to_numpy()
in_any = (c["mirna_labelled"] | c["rbp_labelled"]).to_numpy()
print(f"'neither' circRNAs: {nb.sum()} | not in ENCORI at all: {int((nb & ~in_any).sum())} "
      f"| in ENCORI but no edge passing the evidence rule: {int((nb & in_any).sum())}")

'neither' circRNAs: 256 | not in ENCORI at all: 148 | in ENCORI but no edge passing the evidence rule: 108


## Cell 2, the per-fold builder

In [3]:
import torch, torch_geometric
print(torch.__version__, torch_geometric.__version__, torch.cuda.is_available())

c:\Users\aeasa\anaconda3\envs\circmtl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aeasa\anaconda3\envs\circmtl\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


2.14.1+cu126 2.8.0.post1 True


In [9]:
# 07_graph - Cell 2: per-fold graph builder (numpy index arrays; PyTorch only needed in to_heterodata)
from sklearn.model_selection import GroupShuffleSplit

VAL_FRAC, SUP_FRAC = 0.2, 0.3                     # validation circRNAs; per-epoch supervision share (graph.yaml)
nodes_c = pd.read_csv(G / "nodes_circrna.csv", keep_default_na=False)
nodes_d = pd.read_csv(G / "nodes_disease.csv")
X_raw = np.load(G / "x_circrna.npy")
E = {r: pd.read_csv(G / f"edges_circrna_{r}.csv")[["src", "dst"]].to_numpy(np.int64) for r in ("disease", "mirna", "rbp")}
N = {"circRNA": len(nodes_c), "disease": len(nodes_d),
     "mirna": len(pd.read_csv(G / "nodes_mirna_family.csv")), "rbp": len(pd.read_csv(G / "nodes_rbp.csv"))}
folds = pd.read_csv(D / "folds.csv")
assert folds["circbase_id"].tolist() == nodes_c["circbase_id"].tolist(), "folds.csv out of order"
negs = pd.read_csv(D / "negatives_cda.csv")
C_ID = dict(zip(nodes_c["circbase_id"], nodes_c["circ_idx"]))
D_ID = dict(zip(nodes_d["disease"], nodes_d["disease_id"]))
groups = nodes_c["host_group"].to_numpy()
KNOWN_CD = set(map(tuple, E["disease"]))
AUX = ("mirna", "rbp")

# Auxiliary labels: circRNAs absent from ENCORI are UNKNOWN (never negatives); any ENCORI edge (any evidence) is never a negative
lab = pd.read_csv(D / "circrnas.csv", keep_default_na=False, dtype={"mirna_labelled": bool, "rbp_labelled": bool})
assert lab["circbase_id"].tolist() == nodes_c["circbase_id"].tolist()
LABELLED = {"mirna": lab["mirna_labelled"].to_numpy(), "rbp": lab["rbp_labelled"].to_numpy()}
_seed = pd.read_csv(G / "mirna_families.csv").set_index("mirna_id")["seed"]
_fam = pd.read_csv(G / "nodes_mirna_family.csv").set_index("seed")["fam_idx"]
_rbp = pd.read_csv(G / "nodes_rbp.csv", keep_default_na=False).set_index("rbp")["rbp_idx"]
_am = pd.read_csv(D / "mirna_edges.csv", usecols=["circbase_id", "mirna_id"])
_ar = pd.read_csv(D / "rbp_edges.csv", keep_default_na=False, usecols=["circbase_id", "rbp"])
_am = pd.DataFrame({"c": _am["circbase_id"].map(C_ID), "t": _am["mirna_id"].map(_seed).map(_fam)}).dropna().astype(int)
_ar = pd.DataFrame({"c": _ar["circbase_id"].map(C_ID), "t": _ar["rbp"].map(_rbp)}).dropna().astype(int)
ANY_AUX = {"mirna": set(zip(_am["c"], _am["t"])), "rbp": set(zip(_ar["c"], _ar["t"]))}
for r in AUX:                                           # every working edge is also an any-evidence edge
    assert set(map(tuple, E[r])) <= ANY_AUX[r], f"{r}: working edge missing from the any-evidence set"


def _rows(edges, mask):
    return edges[mask[edges[:, 0]]]

def _val_negatives(pos, train_pos, seed):
    """Degree-matched validation negatives (training disease popularity), never a known positive."""
    rng = np.random.default_rng(seed)
    cnt = np.bincount(train_pos[:, 1], minlength=N["disease"]).astype(float)
    p = cnt / cnt.sum()
    c, d = pos[:, 0].copy(), rng.choice(N["disease"], size=len(pos), p=p)
    for _ in range(100):
        bad = np.array([(a, b) in KNOWN_CD for a, b in zip(c, d)], bool)
        if not bad.any(): break
        d[bad] = rng.choice(N["disease"], size=int(bad.sum()), p=p)
    keep = ~np.array([(a, b) in KNOWN_CD for a, b in zip(c, d)], bool)
    return np.c_[c[keep], d[keep]].astype(np.int64)

def _aux_negatives(pos, train_pos, r, seed):
    """One negative partner per held-out auxiliary edge: same (labelled) circRNA, partner ~ training degree,
    never an ENCORI edge of any evidence."""
    rng = np.random.default_rng(seed)
    cnt = np.bincount(train_pos[:, 1], minlength=N[r]).astype(float)
    p = cnt / cnt.sum()
    c, t = pos[:, 0].copy(), rng.choice(N[r], size=len(pos), p=p)
    known = ANY_AUX[r]
    for _ in range(100):
        bad = np.array([(a, b) in known for a, b in zip(c, t)], bool)
        if not bad.any(): break
        t[bad] = rng.choice(N[r], size=int(bad.sum()), p=p)
    keep = ~np.array([(a, b) in known for a, b in zip(c, t)], bool)
    return np.c_[c[keep], t[keep]].astype(np.int64)

def build_fold(seed, fold):
    """All index arrays for one seed/fold. Test circRNAs are absent from training (inductive)."""
    fcol = folds[f"fold_seed{seed}"].to_numpy()
    test, rest = np.flatnonzero(fcol == fold), np.flatnonzero(fcol != fold)
    tr_i, va_i = next(GroupShuffleSplit(1, test_size=VAL_FRAC, random_state=1000 * seed + fold)
                      .split(rest, groups=groups[rest]))
    train, val = np.sort(rest[tr_i]), np.sort(rest[va_i])
    m = {k: np.isin(np.arange(N["circRNA"]), v) for k, v in (("train", train), ("val", val), ("test", test))}
    fd = {"seed": seed, "fold": fold, "train": train, "val": val, "test": test,
          "mu": X_raw[train].astype(np.float64).mean(0), "sd": X_raw[train].astype(np.float64).std(0) + 1e-8}          # scaler from training circRNAs only
    for r in ("disease",) + AUX:
        for part in ("train", "val", "test"):
            fd[f"{r}_{part}"] = _rows(E[r], m[part])
    fd["neg_val"] = _val_negatives(fd["disease_val"], fd["disease_train"], seed=10_000 + 1000 * seed + fold)
    
    for i, r in enumerate(AUX):                          # frozen auxiliary evaluation negatives (design B)
        pool = np.vstack([fd[f"{r}_train"], fd[f"{r}_val"]])
        fd[f"neg_test_{r}"] = _aux_negatives(fd[f"{r}_test"], pool, r, seed=20_000 + 1000 * seed + 10 * fold + i)
    
    for scheme in ("degree_matched", "random"):
        ng = negs[(negs["seed"] == seed) & (negs["fold"] == fold) & (negs["scheme"] == scheme)]
        fd[f"neg_test_{scheme}"] = np.c_[ng["circbase_id"].map(C_ID), ng["disease"].map(D_ID)].astype(np.int64)
    return fd

def features(fd):
    return ((X_raw.astype(np.float64) - fd["mu"]) / fd["sd"]).astype(np.float32)

def epoch_graph(fd, rng, model):
    """Training step: disjoint message-passing / supervision split of the training edges (fresh each epoch).
    M2 and M3 get the identical graph (same rng draws); M2 simply ignores the auxiliary supervision edges."""
    mp, sup = {}, {}
    rels = ("disease",) if model == "M1" else ("disease",) + AUX
    for r in rels:
        s = rng.random(len(fd[f"{r}_train"])) < SUP_FRAC
        mp[r], sup[r] = fd[f"{r}_train"][~s], fd[f"{r}_train"][s]
    if model != "M1":
        for r in AUX:                                    # validation circRNAs: auxiliary edges are input only
            mp[r] = np.vstack([mp[r], fd[f"{r}_val"]])
    return mp, sup

def inference_graph(fd, stage, model, with_test_aux=True):
    """Disease edges: training circRNAs only, identical at validation and test (the tested graph is the validated graph).
    Auxiliary edges: train + val circRNAs; at test, plus the test circRNAs' auxiliary edges if with_test_aux."""
    rel = {"disease": fd["disease_train"]}                # identical at validation and test
    if model != "M1":
        for r in AUX:
            parts = [fd[f"{r}_train"], fd[f"{r}_val"]] + ([fd[f"{r}_test"]] if stage == "test" and with_test_aux else [])
            rel[r] = np.vstack(parts)
    return rel

REL = {"disease": ("circRNA", "associated_with", "disease"),
       "mirna": ("circRNA", "binds", "mirna"), "rbp": ("circRNA", "bound_by", "rbp")}

def to_heterodata(x, rel):
    """PyTorch Geometric graph; reverse edges are created ONLY from `rel`, so held-out edges never appear reversed."""
    import torch
    from torch_geometric.data import HeteroData
    h = HeteroData()
    h["circRNA"].x = torch.from_numpy(x)
    for r, e in rel.items():
        s, name, t = REL[r]
        h[t].num_nodes = N[t]
        ei = torch.from_numpy(np.ascontiguousarray(e.T)).long()
        h[s, name, t].edge_index = ei
        h[t, "rev_" + name, s].edge_index = ei.flip(0)
    return h

## Cell 3: leakage tests (all must pass before any training)

In [10]:
# 07_graph - Cell 3: leakage tests on every seed and fold (must all pass before training)
def aset(e): return set(map(tuple, e))
results = []
for s in (0, 1, 2):
    for k in range(5):
        fd = build_fold(s, k)
        tr, va, te = set(fd["train"]), set(fd["val"]), set(fd["test"])
        g = lambda ids: set(groups[list(ids)])
        chk = {
            "partition": tr | va | te == set(range(N["circRNA"])) and not (tr & va or tr & te or va & te),
            "groups: test vs train+val": not (g(te) & g(tr | va)),
            "groups: val vs train": not (g(va) & g(tr)),
            "train edges only from train circRNAs": all(set(fd[f"{r}_train"][:, 0]) <= tr for r in REL),
            "no test circRNA in training edges": all(not (set(fd[f"{r}_{p}"][:, 0]) & te) for r in REL for p in ("train", "val")),
            "test negatives on test circRNAs": all(set(fd[f"neg_test_{sc}"][:, 0]) <= te for sc in ("degree_matched", "random")),
            "negatives never positive": not (any(aset(fd[f"neg_test_{sc}"]) & KNOWN_CD for sc in ("degree_matched", "random"))
                                             or aset(fd["neg_val"]) & KNOWN_CD),
            "aux negatives on labelled test circRNAs": all(set(fd[f"neg_test_{r}"][:, 0]) <= (te & set(np.flatnonzero(LABELLED[r]))) for r in AUX),
            "aux negatives never an ENCORI edge": all(not (aset(fd[f"neg_test_{r}"]) & ANY_AUX[r]) for r in AUX),
            "scaler from train only": np.allclose(features(fd)[fd["train"]].mean(0), 0, atol=1e-4),
        }
        rng = np.random.default_rng(0)
        for model in ("M1", "M3"):
            mp, sup = epoch_graph(fd, rng, model)
            chk[f"{model}: mp/sup disjoint"] = all(not (aset(mp[r]) & aset(sup[r])) for r in sup)
            chk[f"{model}: mp has no val/test disease edge"] = not (set(mp["disease"][:, 0]) & (va | te))
            chk[f"{model}: no test circRNA in epoch graph"] = all(not (set(mp[r][:, 0]) & te) for r in mp)
            chk[f"{model}: aux present only if M2/M3"] = (set(mp) == {"disease"}) == (model == "M1")
        vi = inference_graph(fd, "val", "M3")
        chk["val inference: no val/test disease edge"] = not (set(vi["disease"][:, 0]) & (va | te))
        ti = inference_graph(fd, "test", "M3", with_test_aux=True)
        tn = inference_graph(fd, "test", "M3", with_test_aux=False)
        chk["test inference: no test disease edge"] = not (set(ti["disease"][:, 0]) & te)
        chk["test graph = validated graph (disease edges)"] = aset(ti["disease"]) == aset(vi["disease"])   
        chk["aux eval graph: no test aux edge"] = all(not (set(tn[r][:, 0]) & te) for r in AUX)
        chk["disease eval graph: test aux edges added"] = all(len(fd[f"{r}_test"]) == 0 or set(fd[f"{r}_test"][:, 0]) <= set(ti[r][:, 0]) for r in AUX)
        results.append({"seed": s, "fold": k, **chk})
R = pd.DataFrame(results)
failed = [c for c in R.columns[2:] if not R[c].all()]
print(f"{len(R)} seed/fold graphs | {len(R.columns) - 2} checks each | failed checks: {failed if failed else 'none'}")
assert not failed

15 seed/fold graphs | 23 checks each | failed checks: none


## Cell 4: save the frozen per-fold index arrays

In [11]:
# 07_graph - Cell 4: save the frozen per-fold index arrays (the per-epoch split is done in training)
OUTF = G / "folds"; OUTF.mkdir(exist_ok=True)
rows = []
for s in (0, 1, 2):
    for k in range(5):
        fd = build_fold(s, k)
        np.savez_compressed(OUTF / f"seed{s}_fold{k}.npz", **{a: b for a, b in fd.items() if a not in ("seed", "fold")})
        rows.append({"seed": s, "fold": k, "train": len(fd["train"]), "val": len(fd["val"]), "test": len(fd["test"]),
                     "cd_train": len(fd["disease_train"]), "cd_val": len(fd["disease_val"]), "cd_test": len(fd["disease_test"]),
                     "neg_val": len(fd["neg_val"]), "mirna_train": len(fd["mirna_train"]), "rbp_train": len(fd["rbp_train"]),
                     "mirna_test": len(fd["mirna_test"]), "rbp_test": len(fd["rbp_test"]),
                     "cold_disease_pos": int((~np.isin(fd["disease_test"][:, 1], fd["disease_train"][:, 1])).sum()),
                     "cold_mirna_pos": int((~np.isin(fd["mirna_test"][:, 1], np.r_[fd["mirna_train"][:, 1], fd["mirna_val"][:, 1]])).sum()),
                     "cold_rbp_pos": int((~np.isin(fd["rbp_test"][:, 1], np.r_[fd["rbp_train"][:, 1], fd["rbp_val"][:, 1]])).sum())})
S = pd.DataFrame(rows); S.to_csv(OUTF / "fold_summary.csv", index=False)
print(S.to_string(index=False))

 seed  fold  train  val  test  cd_train  cd_val  cd_test  neg_val  mirna_train  rbp_train  mirna_test  rbp_test  cold_disease_pos  cold_mirna_pos  cold_rbp_pos
    0     0   1020  247   315      1391     319      427      319         4992      15703        1525      4206                15               0             2
    0     1   1007  260   315      1375     334      428      334         4492      14211        1543      4591                18               2            20
    0     2   1023  247   312      1386     324      427      324         4422      13781        1575      4736                19               5             5
    0     3   1009  250   323      1381     328      428      328         4919      14795        1135      3967                20               5             2
    0     4   1017  248   317      1346     364      427      364         4456      13784        1762      5078                18               9            14
    1     0   1012  250   320      1408 

In [ ]:
from circmtl.graph import GraphData

: 